### Day 6 Assignment: Medallion Architecture & Lakeflow 

### Basic Tasks 

#### 1. Create Bronze Table
Created the Bronze table `dev.bronze.sale_tbl` using the `ingestion_to_bronze` notebook.

#### 2. Build Silver Table
Created the Silver table `dev.silver.sale_cln_tbl` using the `bronze_to_silver` notebook.

#### 3. Build Gold Table
Created the Gold table `dev.gold.daily_sales_summary`, using the `silver_to_gold` notebook containing daily sales metrics such as total revenue, customers, units sold, and discounts.

### Intermediate Tasks 

#### 4. Bronze/Silver/Gold Pipeline and Primary Consumers

The pipeline follows the Medallion Architecture, where data moves through three layers:

- **Bronze:** Contains raw ingested data with minimal transformations. The primary consumers are **data engineers**, who use it for ingestion, troubleshooting, and data recovery.
- **Silver:** Contains cleaned, validated, and standardized data. The primary consumers are **data analysts and engineers**, who use it for analysis and further transformations.
- **Gold:** Contains business-ready aggregated data such as `daily_sales_summary`. The primary consumers are **data analysts and executives**, who use it for dashboards, KPIs, and business reporting.

**Pipeline flow:** Bronze → Silver → Gold

#### 5. Lakeflow Designer's visual
Created the `dev.silver.sales_clean_designer` table using Lakeflow Designer with visual transformations for removing null values, duplicates, and the ingestion_time column.

![image_1787736867662.png](./image_1787736867662.png "image_1787736867662.png")

#### 6. Lakeflow Job
Created a Lakeflow Job to chain the Bronze, Silver, and Gold tasks with dependencies. The job ran successfully, completing all three tasks in sequence.

![image_1787738079605.png](./image_1787738079605.png "image_1787738079605.png")

###  Advanced Tasks 

#### 7. Gold layer table for the inventory team

In [0]:
%sql
CREATE OR REPLACE TABLE dev.gold.products_sale_summary
USING DELTA AS
SELECT product_id, 
       SUM(quantity) AS total_quantity_sold,
       SUM(total_amount) AS total_revenue,
       SUM(discount_amount) AS total_discounts
FROM dev.silver.sale_cln_tbl
GROUP BY product_id

The Silver table contains transaction-level data, while this Gold table provides a reusable product-level summary for the **inventory team**. Keeping this aggregation in Gold avoids repeated calculations, provides consistent KPIs, and makes the data easier and faster for downstream dashboards and reports.

#### 8. Data Plane vs Control Plane Design

In this pipeline, the actual data processing for the Bronze, Silver, and Gold layers should run in the **customer's data plane**. This includes reading source data, performing transformations, running Spark jobs, and writing the resulting Delta tables. Keeping data processing in the customer's cloud environment provides better control over data access and helps keep sensitive data within the customer's security boundary.

The **Databricks control plane** should primarily handle platform-level services such as job orchestration, scheduling, workspace management, metadata, and monitoring. It should not be responsible for directly processing the customer's raw business data.

From a network and security perspective, the setup should use private connectivity where required, restrict access using IAM and Unity Catalog permissions, and follow least-privilege principles. Network rules should allow only the required communication between the Databricks workspace, data sources, and storage. Credentials and secrets should be managed securely rather than being stored in notebooks or code.

This separation means that the security review should focus on data movement between the control plane, data plane, storage, and external data sources, along with identity, network access, encryption, and permission controls.

#### 9. Gold Table Analysis and Data Quality

1. Query for the Gold table

In [0]:
%sql
SELECT
    order_date,
    total_revenue,
    total_orders,
    total_customers,
    total_products,
    total_units,
    total_discounts
FROM dev.gold.daily_sales_summary
ORDER BY order_date;

In [0]:
%sql

SELECT
    ROUND(SUM(total_revenue), 2) AS total_revenue,
    SUM(total_orders) AS total_orders,
    SUM(total_units) AS total_units,
    SUM(total_discounts) AS total_discounts,
    COUNT(DISTINCT order_date) AS sales_days
FROM dev.gold.daily_sales_summary;

2. Identify a data-quality issue

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT_IF(order_id IS NULL) AS null_order_ids,
    COUNT_IF(customer_id IS NULL) AS null_customer_ids,
    COUNT_IF(product_id IS NULL) AS null_product_ids,
    COUNT_IF(total_amount IS NULL) AS null_amounts
FROM dev.bronze.sale_tbl;

In [0]:
%sql
-- If Bronze contains nulls but Silver has 0 nulls, you have a clear data-quality improvement that you can trace to your Silver transformation.
SELECT
    COUNT(*) AS total_rows,
    COUNT_IF(order_id IS NULL) AS null_order_ids,
    COUNT_IF(customer_id IS NULL) AS null_customer_ids,
    COUNT_IF(product_id IS NULL) AS null_product_ids,
    COUNT_IF(total_amount IS NULL) AS null_amounts
FROM dev.silver.sale_cln_tbl;

Created a simple analysis using the `dev.gold.daily_sales_summary` table to view daily revenue, orders, customers, products, units sold, and discounts. The Gold table can be used directly for dashboard KPIs and daily sales trends.

As a data-quality check, null values were identified in the Bronze sales data. These null records were handled during the Silver transformation, resulting in a cleaned Silver table with the required fields populated. This shows how a data-quality issue originating in Bronze was addressed before the data reached the Gold layer.

The Gold layer therefore provides a reliable, business-ready dataset for reporting and dashboarding while the underlying quality issue is handled earlier in the pipeline.